# OLMo 7B: BBC continuation on Colab A100

Run the cells **in order in a fresh A100 80 GB, high-RAM Colab session**. This is full-parameter training, not LoRA: BF16 computation, FP32 accumulated gradients and CPU FP32 master AdamW, 4096-token blocks, accumulation 16, three epochs. It requires **140 GiB available host RAM**, despite lower GPU memory use.

Uses exactly the LLaDA experiment's even-indexed BBC articles from March-June 2025 at the frozen dataset revision. Both training articles and clean odd controls are verified against the canonical LLaDA token-file hashes before native retokenization. 3,930,266 native training tokens, 960 blocks/epoch, **180 total updates**.

**Storage:** no periodic saves, no epoch model saves, no full-model upload test, no optimizer snapshots. Default output is **one final model, approximately 13.78 GB**, plus small text logs/configuration/diagnostics/methods. Large caches and staging stay on the Colab VM. Set `SAVE_FINAL_MODEL=False` in section 4 for text-only output and no durable trained weights. Nothing large is written to Windows.

The measured compute-only projection is about 2.9 hours; initialization, diagnostics and the single final upload add time. Colab can still terminate a runtime. This final-only notebook cannot recover trained weights if the session ends before that upload completes. Do not run both notebooks simultaneously in the same runtime.

**OLMo context:** 4096 extends beyond the original 2048-token pretraining context. The native configuration and RoPE remain unchanged; this matches the tested probe and is documented as a comparison limitation.

Activation checkpointing is retained to reduce RAM: it recomputes activations and writes no checkpoint files.

## 1. Check hardware and local disk

In [ ]:
import os,sys,json,shutil,subprocess
from pathlib import Path
from datetime import datetime,timezone
import torch
assert Path('/content').is_dir(), 'Run this in Google Colab.'
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'Select a BF16-capable GPU.'
gpu=torch.cuda.get_device_properties(0)
memory=dict(line.split(':',1) for line in Path('/proc/meminfo').read_text().splitlines())
available=int(memory['MemAvailable'].split()[0])*1024/2**30
free_disk=shutil.disk_usage('/content').free/2**30
print(json.dumps(dict(gpu=gpu.name,vram_gib=gpu.total_memory/2**30,available_host_gib=available,free_colab_disk_gib=free_disk),indent=2))
assert gpu.total_memory>=79*2**30,'Select A100 80 GB.'
assert available>=140,'Need at least 140 GiB AVAILABLE host RAM.'
assert free_disk>=55,'Need 55 GiB free local disk for source weights and optional final staging.'
os.environ.update(HF_HUB_CACHE='/content/ar-ft-cache/hub',HF_DATASETS_CACHE='/content/ar-data-cache',
                  HF_HUB_DISABLE_XET='1',TOKENIZERS_PARALLELISM='false',USE_TF='0',USE_FLAX='0')

## 2. Authorize and mount Drive
Approve both prompts. No Hugging Face login is required for these public base models.

In [ ]:
from google.colab import auth,drive
auth.authenticate_user()
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/LLaDA-ft')
for folder in ['runs','logs']:
    (ROOT/folder).mkdir(parents=True,exist_ok=True)

## 3. Install the compatible environment
Uses Colab’s installed PyTorch/CUDA; constrains PyTorch against replacement.

In [ ]:
VENV=Path('/content/ar-ft-venv')
if not (VENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(VENV)],check=True)
PYTHON=str(VENV/'bin/python')
Path('/content/ar-torch-constraint.txt').write_text('torch=='+str(torch.__version__)+'\n')
packages=['transformers==4.46.3','huggingface-hub==0.36.0','tokenizers==0.20.3',
          'accelerate==1.15.0','datasets==4.8.5','safetensors==0.8.0','sentencepiece==0.2.2',
          'psutil','google-api-python-client','google-auth-httplib2']
subprocess.run([PYTHON,'-m','pip','install','--disable-pip-version-check','--constraint','/content/ar-torch-constraint.txt',*packages],check=True)
subprocess.run([PYTHON,'-c','import torch,transformers,accelerate,datasets,psutil,safetensors; import google.auth,googleapiclient; print("AR_IMPORTS_OK",torch.__version__,transformers.__version__)'],check=True)

## 4. Storage choice and frozen training configuration
`SAVE_FINAL_MODEL=True` keeps one final trained model. Set it to `False` for text-only output. Preserve the numerical settings for this comparison. An existing run name is never overwritten.

In [ ]:
SAVE_FINAL_MODEL = True

if 'RUN_NAME' not in globals():
    RUN_NAME = 'olmo7b_bbc_final_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
CONFIG = {'seed': 42, 'epochs': 3, 'sequence_length': 4096, 'microbatch': 1, 'accumulation': 16, 'gradient_dtype': 'float32', 'model_dtype': 'bfloat16', 'master_weight_dtype': 'float32', 'optimizer_state_dtype': 'float32', 'optimizer': 'AdamW', 'optimizer_device': 'cpu', 'peak_learning_rate': 1e-05, 'final_learning_rate': 1e-06, 'warmup_fraction': 0.05, 'decay_fraction': 0.1, 'betas': [0.9, 0.999], 'epsilon': 1e-08, 'weight_decay': 0.1, 'weight_decay_parameter_groups': 'all trainable parameters, including norms and biases if present', 'gradient_clip_norm': 1.0, 'random_length_probability': 0.01, 'random_length_min': 1, 'tail_policy': 'Unpadded final short block; retain all source tokens', 'packing': 'Frozen source-order article content plus EOS, no BOS or chat template; shuffle packed blocks each epoch', 'checkpointing': 'native whole decoder layer non-reentrant', 'cpu_threads': 12, 'diagnostic_blocks_per_group': 8, 'diagnostic_mask_draws_per_block': 0, 'checkpoint_epochs': [], 'checkpoint_format': 'One final BF16 safetensors model only; verified Drive API upload', 'save_optimizer_state': False, 'resume_supported': False, 'methods_filename': 'AR_FINAL_ONLY_NOTEBOOK_METHOD.md', 'checkpoint_backend': 'drive_api', 'checkpoint_every_updates': 0, 'verify_full_checkpoint_before_training': False, 'model_id': 'allenai/OLMo-7B-hf', 'model_revision': '11fb3186a2e4f681edea621fa8b4345147a9db6a', 'objective': 'causal next-token CE; per-sequence target mean then mean over actual accumulation window; length-1 contributes differentiable zero', 'attention': 'native causal SDPA', 'expected_tokens': 3930266, 'expected_blocks_per_epoch': 960, 'expected_updates_per_epoch': 60, 'expected_total_updates': 180, 'warmup_updates': 9, 'decay_updates': 18, 'frozen_article_file_sha256': '8f22ff46d5dfc7d85a2c58d598c72252b754a335514e3bd02ca485a18b87e91e', 'expected_native_token_file_sha256': '1d9f7b504c18921f6c4bd4461ae6df5556cd4e39e8f3707da75eea9ceb84bcb9', 'native_eos_token_id': 50279, 'context_extrapolation': True, 'original_context': 2048, 'status': 'READY_TO_RUN_NOT_STARTED', 'checkpoint_validation': 'Validate AR native state tensor names/shapes/dtypes and index; server checksums; completion marker last', 'diagnostic_passes_per_block': 1, 'storage_plan': 'No model outputs before final; one optional final model plus small text; no automatic deletion', 'model_key': 'olmo7b', 'save_final_model': True, 'expected_parameters': 6888095744, 'expected_control_tokens': 2609034}
CONFIG.update(run_name=RUN_NAME,save_final_model=SAVE_FINAL_MODEL)
assert not (ROOT/'runs'/RUN_NAME).exists(),'Run already started; do not restart under this name.'
CONFIG_PATH=Path('/content/ar_notebook_config.json')
CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
print(json.dumps(CONFIG,indent=2))

## 5. Write the standalone source files
All implementation is embedded here; no Windows files or previous runtime are required.

In [ ]:
%%writefile /content/ar_training_utils.py
import torch
import time
import math
import hashlib
import json
from pathlib import Path

def file_sha(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f,'sha256').hexdigest()

def learning_rate(step,total,config):
    assert 1<=step<=total
    warm=math.ceil(config['warmup_fraction']*total)
    decay=math.ceil(config['decay_fraction']*total)
    peak=config['peak_learning_rate']
    if step<=warm:
        return peak*step/warm
    if step<=total-decay:
        return peak
    return peak+(config['final_learning_rate']-peak)*(step-(total-decay))/decay

def validate_checkpoint(path,expected):
    from safetensors import safe_open
    found={}
    locations={}
    shard_files=sorted(path.glob('*.safetensors'))
    if not shard_files:
        raise RuntimeError('No checkpoint shards')
    for p in shard_files:
        with safe_open(str(p),framework='pt',device='cpu') as f:
            for key in f.keys():
                if key in found:
                    raise RuntimeError(f'Duplicate checkpoint key: {key}')
                s=f.get_slice(key)
                found[key]=(list(s.get_shape()),s.get_dtype())
                locations[key]=p.name
    assert found==expected,'Checkpoint tensor headers differ from model state'
    index=path/'model.safetensors.index.json'
    if index.exists():
        weight_map=json.loads(index.read_text())['weight_map']
        assert set(weight_map)==set(found)
        assert weight_map==locations,'Checkpoint index points keys to incorrect shards'
        assert all(Path(name).name==name and (path/name).is_file() for name in weight_map.values())
    return dict(total_weight_bytes=sum(p.stat().st_size for p in shard_files),
        tensor_count=len(found),files={p.name:dict(bytes=p.stat().st_size,sha256=file_sha(p))
            for p in sorted(path.iterdir()) if p.is_file() and p.name!='COMPLETE.json'})

class CpuMasterAdamW:
    def __init__(self,parameters,config):
        self.params=list(parameters)
        self.buffers={}
        self.masters=[torch.nn.Parameter(p.detach().to(device='cpu',dtype=torch.float32,copy=True)) for p in self.params]
        self.optimizer=torch.optim.AdamW(self.masters,lr=config['peak_learning_rate'],
            betas=tuple(config['betas']),eps=config['epsilon'],weight_decay=config['weight_decay'],foreach=False,fused=False)
        self.clip=config['gradient_clip_norm']
        self.handles=[p.register_post_accumulate_grad_hook(self._accumulate) for p in self.params]

    def _accumulate(self,p):
        with torch.no_grad():
            if id(p) not in self.buffers:
                self.buffers[id(p)]=torch.zeros_like(p,dtype=torch.float32)
            self.buffers[id(p)].add_(p.grad)
            p.grad=None

    def begin(self):
        self.optimizer.zero_grad(set_to_none=True)
        for p in self.params:
            p.grad=None
        for b in self.buffers.values():
            b.zero_()

    def step(self,lr):
        for group in self.optimizer.param_groups:
            group['lr']=lr
        t0=time.perf_counter()
        for p,m in zip(self.params,self.masters):
            assert id(p) in self.buffers,'Missing parameter gradient'
            m.grad=self.buffers[id(p)].detach().to(device='cpu',dtype=torch.float32,copy=True)
        grad_copy=time.perf_counter()-t0
        t1=time.perf_counter()
        norm=torch.nn.utils.clip_grad_norm_(self.masters,self.clip,foreach=False)
        if not torch.isfinite(norm) or norm<=0:
            raise RuntimeError(f'Invalid accumulated gradient norm: {norm}')
        old=self.masters[0].flatten()[:8192].detach().clone()
        self.optimizer.step()
        changed=bool((self.masters[0].flatten()[:8192].detach()!=old).any())
        if not changed:
            raise RuntimeError('FP32 master parameter probe did not change')
        cpu_step=time.perf_counter()-t1
        t2=time.perf_counter()
        with torch.no_grad():
            for p,m in zip(self.params,self.masters):
                p.copy_(m)
        if self.params[0].is_cuda:
            torch.cuda.synchronize()
        return dict(gradient_norm=float(norm),master_parameter_changed=changed,
            gradient_copy_seconds=grad_copy,cpu_optimizer_seconds=cpu_step,
            weight_copy_seconds=time.perf_counter()-t2)

def ar_loss(logits, ids):
    import torch
    if ids.shape[1] < 2:
        return logits[..., 0].float().sum() * 0.0
    return torch.nn.functional.cross_entropy(logits[:, :-1, :].float().reshape(-1, logits.shape[-1]),
                                             ids[:, 1:].reshape(-1), reduction='mean')

In [ ]:
%%writefile /content/drive_checkpoint_upload.py
"""Resumable Drive API uploads, with server-side size and MD5 verification.

Upload session URLs stay in memory and are never printed or stored on Drive.
Existing named files are reused only when content checksums match.
"""
from pathlib import Path
import hashlib
import io
import json
import time

FOLDER='application/vnd.google-apps.folder'
FIELDS='id,name,size,md5Checksum,sha256Checksum,webViewLink,parents,appProperties'


def quoted(value):
    return str(value).replace('\\','\\\\').replace("'","\\'")


def hashes(path):
    md5=hashlib.md5()
    sha=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):
            md5.update(chunk);sha.update(chunk)
    return md5.hexdigest(),sha.hexdigest()


class DriveCheckpointStore:
    def __init__(self,service=None):
        if service is None:
            import google.auth
            from googleapiclient.discovery import build
            credentials,_=google.auth.default()
            service=build('drive','v3',credentials=credentials,cache_discovery=False)
        self.service=service

    def children(self,parent,name=None):
        query=f"'{quoted(parent)}' in parents and trashed=false"
        if name is not None:
            query+=f" and name='{quoted(name)}'"
        result=[]
        token=None
        while True:
            response=self.service.files().list(q=query,fields=f'nextPageToken,files({FIELDS},mimeType)',
                pageSize=1000,pageToken=token).execute(num_retries=5)
            result.extend(response.get('files',[]))
            token=response.get('nextPageToken')
            if not token:
                return result

    def folder(self,parent,name,create=True):
        matches=[f for f in self.children(parent,name) if f['mimeType']==FOLDER]
        if len(matches)>1:
            raise RuntimeError(f'Ambiguous Drive folder name: {name}')
        if matches:
            return matches[0]['id']
        if not create:
            return None
        return self.service.files().create(body=dict(name=name,mimeType=FOLDER,parents=[parent]),fields='id').execute(num_retries=5)['id']

    def ensure_path(self,parts):
        parent='root'
        for part in parts:
            parent=self.folder(parent,part)
        return parent

    def write_state(self,parent,name,data):
        """Overwrite only the small API-owned run-state mirror, not model artifacts."""
        from googleapiclient.http import MediaIoBaseUpload
        existing=self.children(parent,name)
        if len(existing)>1:
            raise RuntimeError('Ambiguous state mirror')
        media=MediaIoBaseUpload(io.BytesIO(json.dumps(data,indent=2).encode()),mimetype='application/json',resumable=False)
        if existing:
            if existing[0].get('appProperties',{}).get('codex_role')!='run_state':
                raise RuntimeError('Refusing to overwrite a state file not created by this uploader')
            return self.service.files().update(fileId=existing[0]['id'],media_body=media,fields=FIELDS).execute(num_retries=5)
        return self.service.files().create(body=dict(name=name,parents=[parent],appProperties={'codex_role':'run_state'}),
            media_body=media,fields=FIELDS).execute(num_retries=5)

    def delete_own_periodic(self,receipt):
        """Collect only a previous periodic folder whose children match our upload receipt."""
        folder=receipt['folder_id']
        expected={v['id'] for v in receipt['files'].values()}
        actual={v['id'] for v in self.children(folder)}
        if actual!=expected:
            raise RuntimeError('Periodic snapshot has unexpected children; automatic cleanup refused')
        for file_id in expected:
            self.service.files().delete(fileId=file_id).execute(num_retries=5)
        self.service.files().delete(fileId=folder).execute(num_retries=5)

    def upload(self,path,parent,progress=None):
        from googleapiclient.http import MediaFileUpload
        path=Path(path)
        size=path.stat().st_size
        md5,sha=hashes(path)
        existing=[f for f in self.children(parent,path.name) if f.get('mimeType')!=FOLDER]
        if len(existing)>1:
            raise RuntimeError(f'Ambiguous Drive file name: {path.name}')
        if existing:
            remote=existing[0]
            if int(remote.get('size',-1))==size and remote.get('md5Checksum')==md5:
                return dict(**remote,verified_md5=md5,local_sha256=sha,reused=True)
            raise RuntimeError(f'Refusing to overwrite different content: {path.name}')
        media=MediaFileUpload(str(path),mimetype='application/octet-stream',chunksize=64*1024*1024,resumable=True)
        request=self.service.files().create(body=dict(name=path.name,parents=[parent],appProperties={'sha256':sha}),
            media_body=media,fields=FIELDS)
        response=None
        try:
            while response is None:
                status,response=request.next_chunk(num_retries=8)
                if status is not None and progress:
                    progress(path.name,status.resumable_progress,size)
        except Exception as error:
            status=getattr(getattr(error,'resp',None),'status',None)
            raise RuntimeError(f'Upload failed for {path.name}: {type(error).__name__}, HTTP {status}; resumable URL withheld') from None
        remote=self.service.files().get(fileId=response['id'],fields=FIELDS).execute(num_retries=5)
        if int(remote.get('size',-1))!=size or remote.get('md5Checksum')!=md5:
            raise RuntimeError(f'Drive verification failed for {path.name}')
        if remote.get('sha256Checksum') and remote['sha256Checksum']!=sha:
            raise RuntimeError(f'Drive SHA256 mismatch for {path.name}')
        return dict(**remote,verified_md5=md5,local_sha256=sha,reused=False)

    def upload_directory(self,path,parent,progress=None):
        path=Path(path)
        if not (path/'COMPLETE.json').is_file():
            raise RuntimeError('Local checkpoint must be validated before upload')
        remote_files={}
        for p in sorted(path.iterdir()):
            if p.is_file() and p.name!='COMPLETE.json':
                remote_files[p.name]=self.upload(p,parent,progress)
        # Completion marker is uploaded only after every shard and sidecar is verified.
        remote_files['COMPLETE.json']=self.upload(path/'COMPLETE.json',parent,progress)
        return dict(folder_id=parent,folder_url=f'https://drive.google.com/drive/folders/{parent}',files=remote_files,
            verified=True,total_bytes=sum(int(v.get('size',0)) for v in remote_files.values()))


def inspect_project():
    store=DriveCheckpointStore()
    project=store.folder('root','LLaDA-ft',create=False)
    if not project:
        raise RuntimeError('Project folder not found via Drive API')
    print('PROJECT_FOLDER_ID',project)
    print('QUOTA',json.dumps(store.service.about().get(fields='storageQuota').execute()))
    def walk(parent,parts,depth):
        for item in store.children(parent):
            p=parts+[item['name']]
            if item['mimeType']==FOLDER:
                if depth:
                    walk(item['id'],p,depth-1)
            else:
                print('FILE',json.dumps(dict(path='/'.join(p),id=item['id'],size=item.get('size'),md5=item.get('md5Checksum'))))
                if item['name']=='status.json' and 'bbc_fp32_3ep_v1' in p:
                    raw=store.service.files().get_media(fileId=item['id']).execute(num_retries=5)
                    print('API_RUN_STATUS',raw.decode('utf-8'))
    for name in ['checkpoints','runs']:
        folder=store.folder(project,name,create=False)
        if folder:
            walk(folder,[name],3)
        else:
            print('ABSENT_FOLDER',name)


if __name__=='__main__':
    inspect_project()

In [ ]:
%%writefile /content/prepare_ar_notebook_data.py
"""Reproduce the frozen articles/controls and download only this AR base locally."""
import os
os.environ.update(HF_HUB_CACHE='/content/ar-ft-cache/hub',HF_DATASETS_CACHE='/content/ar-data-cache',
                  HF_HUB_DISABLE_XET='1',TOKENIZERS_PARALLELISM='false',USE_TF='0',USE_FLAX='0')
import gc
import hashlib
import io
import json
from pathlib import Path
import re
import sys
import numpy as np
from huggingface_hub import HfApi,snapshot_download
from transformers import AutoTokenizer

ROOT=Path('/content/drive/MyDrive/LLaDA-ft')
DATA_REV='8dd1ecdc92ac43f9c04a3da3e945537dbb08179b'
LLADA_REV='0f2787f2d87eac5eed8a087d5ecd24277e6255b2'
ARTICLE_SHA='8f22ff46d5dfc7d85a2c58d598c72252b754a335514e3bd02ca485a18b87e91e'


def digest(text):
    return hashlib.sha256(re.sub(r'\s+',' ',text).strip().encode()).hexdigest()


def encode(tokenizer,texts):
    batches=tokenizer(texts,add_special_tokens=False,return_attention_mask=False,return_token_type_ids=False)['input_ids']
    return np.asarray([token for ids in batches for token in ids+[tokenizer.eos_token_id]],dtype=np.int32)


def array_sha(array):
    data=io.BytesIO();np.save(data,array)
    return hashlib.sha256(data.getvalue()).hexdigest()


def source_rows():
    selected_path=ROOT/'datasets/bbc_even_after_2025_02.jsonl'
    odd_path=ROOT/'datasets/bbc_odd_controls.jsonl'
    if selected_path.exists() and odd_path.exists():
        assert hashlib.sha256(selected_path.read_bytes()).hexdigest()==ARTICLE_SHA,'Frozen article file differs; refusing to train'
        selected=[json.loads(line) for line in selected_path.open()]
        odd=[json.loads(line) for line in odd_path.open()]
    else:
        import datetime
        from datasets import load_dataset
        selected=[];odd=[];filtered=0
        for month in ['2025-03','2025-04','2025-05','2025-06']:
            dataset=load_dataset('RealTimeData/bbc_news_alltime',month,split='train',revision=DATA_REV)
            for source_i,row in enumerate(dataset):
                date=str(row.get('published_date') or '')[:10]
                try:
                    date=datetime.date.fromisoformat(date).isoformat()
                except ValueError:
                    continue
                if date<'2025-03-01':
                    continue
                idx=filtered;filtered+=1
                record=dict(filtered_index=idx,source_config=month,source_row=source_i,
                    published_date=date,title=row.get('title'),link=row.get('link'),content=row.get('content') or '')
                (odd if idx%2 else selected).append(record)
            print('PINNED_MONTH_READY',month,flush=True)
        assert filtered==10643
        canonical=''.join(json.dumps(row,ensure_ascii=False)+'\n' for row in selected).encode()
        assert hashlib.sha256(canonical).hexdigest()==ARTICLE_SHA,'Rebuilt article bytes differ from frozen source'
    assert [row['filtered_index'] for row in selected]==list(range(0,10643,2))
    assert [row['filtered_index'] for row in odd]==list(range(1,10643,2))
    assert all(row['published_date']>='2025-03-01' for row in selected+odd)
    train=[row['content'] for row in selected if row['content'].strip()]
    assert len(train)==5315
    hashes={digest(text) for text in train}
    assert len(hashes)==4236
    clean=[row['content'] for row in odd if row['content'].strip() and digest(row['content']) not in hashes]
    assert len(clean)==3345
    return train,clean


def main(config_path):
    config=json.loads(Path(config_path).read_text())
    output=Path('/content/ar-inputs')/config['model_key'];output.mkdir(parents=True,exist_ok=True)
    train,controls=source_rows()
    reference=AutoTokenizer.from_pretrained('GSAI-ML/LLaDA-8B-Base',revision=LLADA_REV,trust_remote_code=True)
    reference_tokens=encode(reference,train)
    assert len(reference_tokens)==3941989 and array_sha(reference_tokens)=='f5e29db9e7b1459bf030cac1567f77f6595da59acdfaf010ab977b21fa0254df'
    del reference_tokens
    reference_tokens=encode(reference,controls)
    assert len(reference_tokens)==2617971 and array_sha(reference_tokens)=='3a6968777b4ae68e4ed1f89b3309ab4e1133a43a2a659bf7fe561dc70416e27d'
    del reference_tokens,reference
    gc.collect()
    print('EXACT_LLADA_ARTICLES_AND_CONTROLS_VERIFIED',flush=True)
    tokenizer=AutoTokenizer.from_pretrained(config['model_id'],revision=config['model_revision'],use_fast=True)
    assert tokenizer.eos_token_id==config['native_eos_token_id']
    files={}
    for filename,texts,count in [('train_tokens.npy',train,config['expected_tokens']),
                                ('control_tokens.npy',controls,config['expected_control_tokens'])]:
        tokens=encode(tokenizer,texts)
        assert len(tokens)==count
        path=output/filename;np.save(path,tokens)
        files[filename]=dict(bytes=path.stat().st_size,sha256=hashlib.sha256(path.read_bytes()).hexdigest())
        if filename=='train_tokens.npy':
            assert files[filename]['sha256']==config['expected_native_token_file_sha256']
        del tokens
    del train,controls
    gc.collect()
    info=HfApi().model_info(config['model_id'],revision=config['model_revision'])
    assert info.sha==config['model_revision']
    names=[item.rfilename for item in info.siblings if '/' not in item.rfilename]
    weights=[name for name in names if name.endswith('.safetensors')]
    assert weights,'Require native safetensors weights'
    allowed=weights+[name for name in names if name.endswith(('.json','.model','.txt','.tiktoken'))]
    model_path=snapshot_download(config['model_id'],revision=config['model_revision'],allow_patterns=allowed,max_workers=4)
    index_path=Path(model_path)/'model.safetensors.index.json'
    if index_path.exists():
        weight_map=json.loads(index_path.read_text())['weight_map']
        assert set(weight_map.values())==set(weights)
        assert all(Path(name).name==name and (Path(model_path)/name).is_file() for name in weight_map.values())
    manifest=dict(model_id=config['model_id'],model_revision=config['model_revision'],model_path=model_path,
        dataset_id='RealTimeData/bbc_news_alltime',dataset_revision=DATA_REV,
        selected_articles=5322,nonempty_train_articles=5315,clean_control_articles=3345,
        empty_train_articles=7,unique_normalized_train_contents=4236,duplicate_train_rows=1079,
        frozen_article_file_sha256=ARTICLE_SHA,train_tokens=config['expected_tokens'],
        clean_control_tokens=config['expected_control_tokens'],token_files=files,
        tokenization='Native tokenizer, no BOS/chat/title; content plus one EOS per nonempty article')
    (output/'data_manifest.json').write_text(json.dumps(manifest,indent=2))
    print('AR_LOCAL_MODEL_AND_DATA_READY',json.dumps(manifest),flush=True)


if __name__=='__main__':
    main(sys.argv[1])

In [ ]:
%%writefile /content/train_ar_bbc.py
"""Matched AR continuation, text diagnostics and one optional final model save."""
import os
os.environ.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false')
import argparse
import datetime
import gc
import hashlib
import importlib.metadata
import json
import math
from pathlib import Path
import shutil
import time
import traceback
import numpy as np
import psutil
import torch
from ar_training_utils import CpuMasterAdamW, ar_loss, learning_rate, validate_checkpoint, file_sha
from drive_checkpoint_upload import DriveCheckpointStore


def diagnostics(model, streams, config, epoch):
    records=[]
    model.eval()
    try:
        with torch.random.fork_rng(devices=[torch.cuda.current_device()]), torch.no_grad():
            for group,tokens in streams.items():
                full=len(tokens)//config['sequence_length']
                positions=np.linspace(0,full-1,min(full,config['diagnostic_blocks_per_group']),dtype=int)
                for block in positions:
                    ids=torch.as_tensor(tokens[block*4096:(block+1)*4096].astype(np.int64),device='cuda')[None,:]
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        logits=model(input_ids=ids,use_cache=False).logits
                        loss=ar_loss(logits,ids)
                    assert torch.isfinite(loss),'Nonfinite diagnostic loss'
                    records.append(dict(group=group,block=int(block),loss=float(loss)))
                    del ids,logits,loss
    finally:
        model.train()
    return dict(epoch=epoch,metric='deterministic native-token next-token CE; not a memorization measure',
                means={g:float(np.mean([r['loss'] for r in records if r['group']==g])) for g in streams},records=records)


def export_final(model,tokenizer,config,run_dir,environment,store,journal,emit):
    """Only large durable output: one final model, after releasing optimizer state."""
    payload=config['expected_parameters']*2
    quota=store.service.about().get(fields='storageQuota').execute(num_retries=5)['storageQuota']
    if quota.get('limit'):
        assert int(quota['limit'])-int(quota.get('usage',0))>=payload+2*2**30,'Insufficient final-save Drive quota'
    assert shutil.disk_usage('/content').free>=payload+2*2**30,'Insufficient local staging space'
    destination=Path('/content/ar-final-staging')/config['run_name']/'final'
    destination.mkdir(parents=True,exist_ok=False)
    emit('final_model_serialization_started')
    dtype_codes={'torch.bfloat16':'BF16','torch.float32':'F32','torch.int64':'I64','torch.bool':'BOOL'}
    expected={key:(list(value.shape),dtype_codes[str(value.dtype)]) for key,value in model.state_dict().items()}
    model.save_pretrained(destination,safe_serialization=True,max_shard_size='2GB')
    tokenizer.save_pretrained(destination)
    shutil.copy2(run_dir/'config.json',destination/'training_config.json')
    shutil.copy2(run_dir/'METHODS.md',destination/'METHODS.md')
    complete=validate_checkpoint(destination,expected)
    complete.update(run=config['run_name'],epoch=config['epochs'],update=config['expected_total_updates'],
                    model_id=config['model_id'],model_revision=config['model_revision'],
                    optimizer_saved=False,exact_training_resume_supported=False,
                    config_sha256=environment['config_sha256'],methods_sha256=environment['methods_sha256'])
    (destination/'COMPLETE.json').write_text(json.dumps(complete,indent=2))
    folder=store.ensure_path(['LLaDA-ft','ar-models',config['run_name'],'final'])
    previous={}
    def progress(name,done,total):
        if done-previous.get(name,0)>=256*2**20 or done==total:
            previous[name]=done
            emit('final_model_upload_progress',file=name,bytes=done,total=total)
    receipt=store.upload_directory(destination,folder,progress)
    receipt.update(epoch=config['epochs'],update=config['expected_total_updates'],run=config['run_name'])
    path=run_dir/'final_upload_receipt.json'
    path.write_text(json.dumps(receipt,indent=2))
    store.upload(path,journal)
    emit('final_model_verified',url=receipt['folder_url'],weight_bytes=complete['total_weight_bytes'])
    assert destination.resolve().is_relative_to(Path('/content/ar-final-staging').resolve())
    shutil.rmtree(destination)
    return receipt['folder_url']


def run(config_path):
    from transformers import AutoModelForCausalLM,AutoTokenizer
    config=json.loads(Path(config_path).read_text())
    name=config['run_name']
    assert Path(name).name==name and name not in ('.','..')
    assert config['epochs']==3 and config['sequence_length']==4096 and config['accumulation']==16
    assert config['checkpoint_epochs']==[] and config['checkpoint_every_updates']==0
    assert not config['verify_full_checkpoint_before_training'] and not config['save_optimizer_state']
    root=Path('/content/drive/MyDrive/LLaDA-ft')
    local=Path('/content/ar-inputs')/config['model_key']
    run_dir=root/'runs'/name
    assert not run_dir.exists(),'Run already exists; refusing to overwrite/restart'
    store=DriveCheckpointStore()
    project=store.folder('root','LLaDA-ft',create=False)
    assert project
    existing_journal=store.folder(project,'run-api-records',create=False)
    if existing_journal:
        prior=store.folder(existing_journal,name,create=False)
        assert prior is None or not store.children(prior,'status_api.json'),'API journal already contains this run'
    run_dir.mkdir(parents=True)
    journal=store.ensure_path(['LLaDA-ft','run-api-records',name])
    state=dict(run=name,status='preparing',epoch=0,update=0,save_final_model=config['save_final_model'])
    started=time.perf_counter()
    def emit(event,**fields):
        record=dict(time_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),event=event,**fields)
        state.update(last_event=event,last_time_utc=record['time_utc'],elapsed_seconds=time.perf_counter()-started)
        with (run_dir/'events.jsonl').open('a') as f:
            f.write(json.dumps(record)+'\n')
        (run_dir/'status.json').write_text(json.dumps(state,indent=2))
        store.write_state(journal,'status_api.json',state)
        print(json.dumps(record),flush=True)
    try:
        torch.set_num_threads(config['cpu_threads'])
        torch.manual_seed(config['seed'])
        torch.backends.cuda.matmul.allow_tf32=True
        assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        assert torch.cuda.get_device_properties(0).total_memory>=79*2**30
        assert psutil.virtual_memory().available>=140*2**30,'Need 140 GiB available host RAM'
        data=json.loads((local/'data_manifest.json').read_text())
        assert data['model_revision']==config['model_revision']
        assert data['train_tokens']==config['expected_tokens']
        for filename,details in data['token_files'].items():
            assert file_sha(local/filename)==details['sha256'] and (local/filename).stat().st_size==details['bytes']
        shutil.copy2(config_path,run_dir/'config.json')
        shutil.copy2('/content/AR_NOTEBOOK_METHOD.md',run_dir/'METHODS.md')
        shutil.copy2(local/'data_manifest.json',run_dir/'data_manifest.json')
        sources=['train_ar_bbc.py','prepare_ar_notebook_data.py','ar_training_utils.py','drive_checkpoint_upload.py']
        for source in sources:
            shutil.copy2(Path('/content')/source,run_dir/source)
        environment=dict(python=__import__('sys').version,
            packages={p:importlib.metadata.version(p) for p in ['torch','transformers','tokenizers','accelerate','safetensors','datasets']},
            gpu=torch.cuda.get_device_name(0),gpu_vram_gib=torch.cuda.get_device_properties(0).total_memory/2**30,
            source_sha256={s:file_sha(Path('/content')/s) for s in sources},
            config_sha256=file_sha(config_path),methods_sha256=file_sha(run_dir/'METHODS.md'),
            model_revision=config['model_revision'],storage='text logs plus optional one final model only')
        (run_dir/'environment.json').write_text(json.dumps(environment,indent=2))
        for filename in ['config.json','METHODS.md','data_manifest.json','environment.json']+sources:
            store.upload(run_dir/filename,journal)
        model=AutoModelForCausalLM.from_pretrained(data['model_path'],torch_dtype=torch.bfloat16,
            device_map={'':0},low_cpu_mem_usage=True,attn_implementation='sdpa')
        model.config.use_cache=False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
        params=list(model.parameters())
        assert sum(p.numel() for p in params)==config['expected_parameters']
        assert all(p.requires_grad and p.dtype==torch.bfloat16 and p.is_cuda for p in params)
        tokenizer=AutoTokenizer.from_pretrained(data['model_path'])
        # Memory-map token arrays on local disk; move only one block onto the GPU.
        streams={g:np.load(local/f,mmap_mode='r') for g,f in [('train','train_tokens.npy'),('clean_odd_controls','control_tokens.npy')]}
        count=math.ceil(len(streams['train'])/4096)
        per_epoch=math.ceil(count/16)
        total=per_epoch*3
        assert (count,per_epoch,total)==(config['expected_blocks_per_epoch'],config['expected_updates_per_epoch'],config['expected_total_updates'])
        baseline=diagnostics(model,streams,config,0)
        p=run_dir/'diagnostics_epoch_000.json';p.write_text(json.dumps(baseline,indent=2));store.upload(p,journal)
        emit('baseline_complete',means=baseline['means'])
        opt=CpuMasterAdamW(params,config)
        state['status']='running'
        emit('training_started',total_updates=total,blocks_per_epoch=count)
        update=0
        for epoch in range(1,4):
            state['epoch']=epoch
            order=np.random.default_rng(config['seed']+epoch).permutation(count)
            crops=np.random.default_rng(config['seed']+10000+epoch)
            emit('epoch_started',epoch=epoch,block_order_sha256=hashlib.sha256(order.tobytes()).hexdigest())
            epoch_tokens=epoch_targets=epoch_crops=0
            for offset in range(0,count,16):
                indices=order[offset:offset+16]
                opt.begin()
                torch.cuda.synchronize();t0=time.perf_counter()
                losses=[];window_tokens=window_targets=window_crops=0
                for block in indices:
                    ids=torch.as_tensor(streams['train'][block*4096:min((block+1)*4096,len(streams['train']))].astype(np.int64),device='cuda')[None,:]
                    if crops.random()<config['random_length_probability']:
                        ids=ids[:,:int(crops.integers(1,ids.shape[1]+1))]
                        window_crops+=1
                    window_tokens+=ids.numel();window_targets+=max(0,ids.numel()-1)
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        logits=model(input_ids=ids,use_cache=False).logits
                        loss=ar_loss(logits,ids)
                    assert torch.isfinite(loss),'Nonfinite training loss'
                    (loss/len(indices)).backward()
                    losses.append(float(loss.detach()))
                    del ids,logits,loss
                torch.cuda.synchronize()
                fb=time.perf_counter()-t0
                update+=1
                lr=learning_rate(update,total,config)
                result=opt.step(lr)
                epoch_tokens+=window_tokens;epoch_targets+=window_targets;epoch_crops+=window_crops
                state.update(update=update,epoch_tokens=epoch_tokens,epoch_targets=epoch_targets)
                emit('optimizer_update',epoch=epoch,update=update,epoch_update=offset//16+1,
                    window_size=len(indices),mean_loss=float(np.mean(losses)),learning_rate=lr,
                    processed_tokens=window_tokens,prediction_targets=window_targets,random_crops=window_crops,
                    forward_backward_seconds=fb,update_seconds=time.perf_counter()-t0,
                    cpu_rss_gib=psutil.Process().memory_info().rss/2**30,
                    gpu_allocated_gib=torch.cuda.memory_allocated()/2**30,**result)
            opt.optimizer.zero_grad(set_to_none=True)
            emit('epoch_training_complete',epoch=epoch,processed_tokens=epoch_tokens,prediction_targets=epoch_targets,random_crops=epoch_crops)
            diag=diagnostics(model,streams,config,epoch)
            p=run_dir/f'diagnostics_epoch_{epoch:03d}.json';p.write_text(json.dumps(diag,indent=2));store.upload(p,journal)
            emit('diagnostics_complete',epoch=epoch,means=diag['means'])
        assert update==total
        # Remove hooks and large CPU/GPU optimizer allocations before serialization.
        for handle in opt.handles:
            handle.remove()
        opt.optimizer.zero_grad(set_to_none=True)
        opt.optimizer.state.clear();opt.buffers.clear()
        del opt
        for p in params:
            p.grad=None
        gc.collect();torch.cuda.empty_cache()
        state['status']='saving_final' if config['save_final_model'] else 'complete'
        emit('optimization_complete',updates=update)
        if config['save_final_model']:
            state['final_model_url']=export_final(model,tokenizer,config,run_dir,environment,store,journal,emit)
            state['final_model_verified']=True
        state['status']='complete'
        emit('training_complete',updates=update,final_model_saved=config['save_final_model'])
    except BaseException as error:
        state.update(status='failed',error=str(error),traceback=traceback.format_exc())
        emit('training_failed',error_type=type(error).__name__,error=str(error))
        raise


if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('--config',required=True)
    run(parser.parse_args().config)

In [ ]:
%%writefile /content/validate_ar_final_only.py
"""CPU checks for final-only saving, integrity failures and matched AR math."""
import os
os.environ.update(USE_TF='0',USE_FLAX='0')
import ast
import hashlib
import json
from pathlib import Path
import shutil
import tempfile
import uuid
import torch
from transformers import LlamaConfig,LlamaForCausalLM
from ar_training_utils import ar_loss,learning_rate,validate_checkpoint
from train_ar_bbc import export_final
from drive_checkpoint_upload import DriveCheckpointStore

torch.set_num_threads(12)
torch.manual_seed(9)
model=LlamaForCausalLM(LlamaConfig(vocab_size=31,hidden_size=16,intermediate_size=32,
    num_hidden_layers=1,num_attention_heads=2,num_key_value_heads=1,max_position_embeddings=64))
ids=torch.randint(0,31,(2,13))
out=model(input_ids=ids,labels=ids,use_cache=False)
torch.testing.assert_close(ar_loss(out.logits,ids),out.loss)
one_logits=torch.randn(1,1,31,requires_grad=True)
zero=ar_loss(one_logits,ids[:1,:1]);zero.backward()
assert zero.item()==0 and torch.count_nonzero(one_logits.grad)==0
for total in (180,198):
    config=dict(warmup_fraction=.05,decay_fraction=.1,peak_learning_rate=1e-5,final_learning_rate=1e-6)
    assert abs(learning_rate(total,total,config)-1e-6)<1e-15
    import math
    assert learning_rate(math.ceil(.05*total),total,config)==1e-5

class FakeService:
    def about(self):return self
    def get(self,**kwargs):return self
    def execute(self,**kwargs):return dict(storageQuota=dict(limit=str(10**12),usage='0'))

class Recorder(DriveCheckpointStore):
    def __init__(self,fail=False):self.service=FakeService();self.calls=[];self.fail=fail
    def ensure_path(self,parts):assert parts[-1]=='final';return 'final-folder'
    def upload(self,path,parent,progress=None):
        self.calls.append(path.name)
        if self.fail and path.name.endswith('.safetensors'):
            raise RuntimeError('Simulated upload failure')
        content=path.read_bytes()
        return dict(id=path.name,name=path.name,size=str(len(content)),
            md5Checksum=hashlib.md5(content).hexdigest(),verified_md5=hashlib.md5(content).hexdigest(),
            local_sha256=hashlib.sha256(content).hexdigest())

class Tokenizer:
    def save_pretrained(self,path):(path/'tokenizer_config.json').write_text('{}')

model=model.to(torch.bfloat16)
with tempfile.TemporaryDirectory() as temp:
    root=Path(temp)
    (root/'config.json').write_text('{}');(root/'METHODS.md').write_text('test protocol')
    config=dict(expected_parameters=sum(p.numel() for p in model.parameters()),
        run_name='unit_final_'+uuid.uuid4().hex,epochs=3,expected_total_updates=198,
        model_id='tiny-test',model_revision='test-revision')
    events=[]
    def emit(event,**kwargs):events.append(event)
    store=Recorder()
    result=export_final(model,Tokenizer(),config,root,dict(config_sha256='config',methods_sha256='methods'),store,'journal',emit)
    assert result.endswith('/final-folder')
    assert store.calls[-2:]==['COMPLETE.json','final_upload_receipt.json']
    assert events[-1]=='final_model_verified'
    assert not (Path('/content/ar-final-staging')/config['run_name']/'final').exists()
    # Failed upload must not claim completion or discard the only local trained model.
    config['run_name']='unit_failure_'+uuid.uuid4().hex
    store=Recorder(fail=True);events.clear()
    try:
        export_final(model,Tokenizer(),config,root,dict(config_sha256='config',methods_sha256='methods'),store,'journal',emit)
    except RuntimeError as error:
        assert 'Simulated' in str(error)
    else:raise AssertionError('Upload failure accepted')
    assert 'COMPLETE.json' not in store.calls and 'final_model_verified' not in events
    staging=Path('/content/ar-final-staging')/config['run_name']/'final'
    assert staging.is_dir()
    assert staging.resolve().is_relative_to(Path('/content/ar-final-staging').resolve())
    shutil.rmtree(staging)

tree=ast.parse(Path('/content/train_ar_bbc.py').read_text())
run=next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name=='run')
calls=[n for n in ast.walk(run) if isinstance(n,ast.Call) and isinstance(n.func,ast.Name) and n.func.id=='export_final']
assert len(calls)==1
for node in ast.walk(run):
    if isinstance(node,(ast.For,ast.While)):
        assert not any(n in calls for n in ast.walk(node)),'Final model export inside a training loop'
guards=[n for n in ast.walk(run) if isinstance(n,ast.If) and ast.unparse(n.test)=="config['save_final_model']"]
assert len(guards)==1 and any(n in calls for n in ast.walk(guards[0]))
print('FINAL_ONLY_CHECKS_PASSED: native AR loss, zero targets, schedules, tiny safetensors validation, marker-last, failed-save retention, one guarded final export outside training loops')

In [ ]:
#@title Write the full comparison and final-only methods document
METHODS = "# Mistral / OLMo final-only notebook protocol\n\nThis notebook retains the matched three-epoch, 4096-token, full-parameter AR continuation protocol below, with these storage and execution changes:\n\n- No periodic model saves, epoch model saves, untrained-base export or full-size upload preflight. No optimizer/master/moment snapshots are written.\n- By default, save exactly one final BF16 model after all three epochs, using local Colab staging, validated safetensors and checksummed resumable Drive API upload with COMPLETE.json last. Approximate final weights: Mistral 14.48 GB; OLMo 13.78 GB. Epoch-1/2 weight comparisons and recovery after a runtime termination are unavailable in this final-only design.\n- `SAVE_FINAL_MODEL=False` instead produces only small text logs, configuration, source code, methods, hashes and diagnostics. It does not produce any durable trained model; all trained weights are lost when the runtime ends.\n- Preserve BF16 computation, FP32 gradient accumulation and CPU FP32 master AdamW state. Do not introduce LoRA, quantization or altered optimizer precision to reduce memory. The tested runtime still needs an A100 80 GB and high host RAM.\n- Keep non-reentrant activation checkpointing: this recomputes activations to reduce RAM and does not save files. Stream one input block onto the GPU; memory-map local token files; release optimizer hooks, master weights, moments and gradient buffers before final serialization. These changes do not change intended training arithmetic.\n- Base weights, Hugging Face/dataset caches, token arrays and serialization staging reside on Colab local disk. No large Windows writes or Drive model cache. Only the optional final model and small text artifacts go to Drive.\n- Preserve baseline/per-epoch text diagnostics. They do not save model weights. Training runs as a blocking notebook cell with live progress; there is no synthetic keepalive, automatic restart or exact optimizer resume.\n- The production schedule uses the model-specific update count: Mistral 198 total updates (10 warmup / 20 decay); OLMo 180 (9 warmup / 18 decay). The 1% random-length policy includes a one-token crop with zero AR target loss, averaged as a real occurrence.\n- Original OLMo remains at unscaled 4096 computation with its native 2048 pretraining-context configuration unchanged. This explicitly extrapolates beyond its original context; a successful feasibility probe does not validate its effect on memorization quality.\n\nThe appended comparison document describes the reference and earlier feasibility protocol. This section overrides its intermediate checkpoint/retention statements for these two notebooks. Neither notebook starts a different model or creates a Colab runtime.\n\n---\n\n# Autoregressive comparison to the completed LLaDA BBC experiment\n\nDate: 26 October 2026. Scope: feasibility probes and a specified continuation protocol; production AR training has not been started.\n\n## Reference experiment\n\nThe primary reference is successful LLaDA run `bbc_fp32_colab_20261026_084128_323830`, completed after three epochs and 183 updates. Its protocol, source hashes, configuration and environment are preserved in Drive. The October 22 notebook run also completed, but is a separate replicate; do not combine the two as a single exposure history.\n\nThe intended comparison changes the predictive objective from masked diffusion to autoregressive next-token prediction while preserving all feasible continuation settings. It cannot literally change only the objective: the requested models have different released weights, tokenizers, parameter counts, attention architectures and original training histories. These are explicit comparison factors, not implementation details to conceal.\n\n## Exact model identities\n\nUse the original base releases, not chat/instruction variants or later model generations. Resolve and record a full immutable revision before any tokenizer/weight download.\n\n| Requested model | Repository | Selection rationale |\n|---|---|---|\n| Llama 2 7B | `meta-llama/Llama-2-7b-hf` | Official original Llama 2 base, Hugging Face format |\n| Llama 3 8B | `meta-llama/Meta-Llama-3-8B` | Official original Llama 3 base; not Llama 3.1/3.2 or Instruct |\n| Mistral-7B | `mistralai/Mistral-7B-v0.1` | First original Mistral 7B base; v0.3 would be another experimental model |\n| OLMo-7B | `allenai/OLMo-7B-hf` | Authors' Transformers-format conversion of the original OLMo-7B; not OLMo-1.7, OLMo-2 or Instruct |\n\nThe OLMo authors identify `OLMo-7B-hf` as the conversion of the original release and recommend it with modern Transformers. Its card's statement that it does not directly support their fine-tuning processes refers to the original OLMo tooling; actual Transformers backward/optimizer feasibility is tested here rather than assumed from that sentence. Sources: [original card](https://huggingface.co/allenai/OLMo-7B/blob/main/README.md), [converted card](https://huggingface.co/allenai/OLMo-7B-hf/blob/main/README.md).\n\nLlama downloads require a Hugging Face login with approved access. Initial anonymous requests returned HTTP 401 for both repositories. The user authenticates directly in Colab; tokens are never placed in scripts, reports or this workspace. Authentication is not a numerical training difference.\n\n## Corpus identity and tokenization\n\nUse the frozen `RealTimeData/bbc_news_alltime` revision `8dd1ecdc92ac43f9c04a3da3e945537dbb08179b`, configurations `2025-03`, `2025-04`, `2025-05`, `2025-06`, sorted in that order. Date-filter publication dates to at least 2025-03-01, assign one global zero-based index in original source order, and select even indices. This is the already completed LLaDA corpus, not an expanded snapshot including any new monthly data.\n\nUse exactly the saved `datasets/bbc_even_after_2025_02.jsonl`: 5,322 rows, indices `0,2,...,10642`. Keep all training duplicates. Omit the same seven empty contents from tokenization, leaving 5,315 nonempty articles. Use only the original article content, with no title, instruction, chat template, BOS, extra textual delimiter, deduplication or normalization of the training text.\n\nThe frozen JSONL currently has SHA-256 `8f22ff46d5dfc7d85a2c58d598c72252b754a335514e3bd02ca485a18b87e91e`. Each probe checks the complete index list, dates and empty-row count and records this file hash. The additional reference-tokenization audit checks that this text reproduces the canonical LLaDA token file. All AR models must use this same file hash.\n\nTokenize each nonempty article with that model's native tokenizer and `add_special_tokens=False`, append exactly one native EOS, concatenate in the frozen article order, then pack into 4096-token blocks. Keep the final short block unpadded. Save a distinct token-file hash and tokenizer/model revision for every model. Reusing LLaDA token IDs in another vocabulary would encode different text and is prohibited.\n\nConsequences of native tokenization:\n\n- Article identity, ordering and nominal three-pass corpus exposure match; token counts, block boundaries, EOS IDs, tail lengths and examples seen together in an update differ.\n- With microbatch 1 and accumulation 16, the number of optimizer updates is computed from each actual block count. Three epochs and the original 183 updates cannot both be held constant when tokenization changes. Preserve three epochs and accumulation 16; report the resulting update counts. Do not truncate or repeat data to force 183.\n- Equal 4096-token context and effective batch do not imply equal bytes/characters of context per model. Weight-decay applications and the number of optimizer steps can differ as a consequence. A separate fixed-update experiment would be a different protocol.\n- Omitting BOS matches the LLaDA continuation procedure, even for a native tokenizer whose default would insert BOS. It is not a claim about the AR model's original pretraining recipe.\n\n## Settings to preserve in full continuation\n\n| Setting | Matched value or policy |\n|---|---|\n| Parameters | All released model parameters trainable; no LoRA, quantization or frozen layers |\n| Model computation | BF16 GPU parameters/computation |\n| Accumulation | FP32 GPU buffers, with each BF16 parameter gradient added then released |\n| CPU optimizer | Independent FP32 master weights, stock PyTorch AdamW and FP32 moments |\n| Optimizer implementation | Exact `CpuMasterAdamW` class extracted from the reviewed LLaDA trainer; no altered update rule |\n| AdamW | betas `(0.9,0.999)`, epsilon `1e-8`, weight decay `0.1` for every trainable parameter including norms/biases |\n| Clipping | One CPU FP32 global gradient norm, limit 1.0, after the accumulation window |\n| Sequence length | 4096, with unpadded final short block; OLMo context issue below |\n| Batch | Microbatch 1; accumulate 16; final window each epoch averaged over its actual size |\n| Epochs | Three; no control-driven stopping or tuning |\n| Length variation | Independent 1% crop per block occurrence to a uniformly sampled integer length 1 through current block length |\n| Packing and shuffle | Frozen source-order packing; shuffled blocks per epoch with NumPy generator seed `42+epoch` |\n| Crop RNG | Separate NumPy generator seed `42+10000+epoch` |\n| Model RNG | PyTorch seed 42; no claim of cross-model identical RNG trajectories |\n| LR peak/final | `1e-5` / `1e-6` |\n| LR schedule | First `ceil(0.05*total_updates)` warmup, steady peak, final `ceil(0.10*total_updates)` linear decay to final rate; count actual model-specific updates |\n| Memory strategy | Whole decoder-layer activation checkpointing, non-reentrant; KV cache disabled |\n| Host execution | 12 CPU threads; same Colab A100 80 GB and high host RAM |\n| TF32 | Enabled as in LLaDA reference |\n| Durable output | BF16 model-only epoch snapshots, tokenizer/configuration, method/config/code/data hashes and upload receipts |\n| Storage mechanism | Serialize/validate on Colab local disk; resumable Drive API uploads; server size/MD5/SHA checks; completion marker last |\n| Resume | No exact optimizer-state resume; new run identifiers required for separately specified restarts |\n\nThe learning rate and AdamW settings are matched to the completed LLaDA continuation experiment, not to each AR model's original pretraining optimizer. Replacing them with the original AR model's defaults would add a confound. For example, the original OLMo card reports a different beta2 and epsilon; those values are not used here. [OLMo pretraining hyperparameters](https://huggingface.co/allenai/OLMo-7B/blob/main/README.md).\n\n## Required objective and attention differences\n\nLLaDA used bidirectional attention, independently sampled corruption per occurrence and inverse-mask-probability weighted, unshifted reconstruction loss. AR uses the released causal attention and uncorrupted input with teacher-forced shifted targets:\n\n`loss(block) = mean(CE(logits[0:T-1].float(), token_ids[1:T]))`.\n\nEach micro-example's mean loss is averaged over the actual accumulation-window size, preserving LLaDA's sequence-mean then window-mean convention. Do not flatten together variable-length examples and change weighting to a token-weighted batch mean. EOS participates as an ordinary next-token target; predictions may cross article EOS boundaries within the concatenated stream, matching the lack of document-isolation masking in the LLaDA continuation.\n\nThe first token of each independent block has no preceding input and is not an AR target. Thus an uncropped block has `T-1` targets, unlike LLaDA's same-position reconstruction opportunities across `T` positions. A length-1 crop has no AR target; use a differentiable zero contribution and preserve its occurrence and denominator. Do not silently redraw it or change the crop minimum to 2. Adding BOS or a preceding-context token to recover every first-token target would change the specified packing/context policy.\n\nNo diffusion mask draws, mask ID, inverse-probability weights or random noising occur for AR. Preserve each native released architecture, RoPE settings, normalization, grouped-query/full-head pattern and vocabulary. Do not convert an AR model to bidirectional attention to imitate LLaDA; that would change the model and predictive task.\n\nHugging Face native AR modules use causal PyTorch SDPA during the probes; the original LLaDA custom module has its own attention implementation. At lengths at most 4096, Mistral's native 4096 sliding window does not exclude any additional in-block causal history. Floating-point kernel arithmetic can differ, so bitwise equivalence is not claimed. [Mistral configuration](https://huggingface.co/mistralai/Mistral-7B-v0.1/blob/main/config.json).\n\n## OLMo context and source-precision limitations\n\nThe original OLMo-7B release has `max_position_embeddings=2048`, RoPE theta 10000 and no configured scaling. Matching LLaDA's 4096 therefore extends OLMo beyond its original pretraining context. The primary probe attempts actual 4096-token forward/backward and CPU-master updates with the native configuration unchanged. A computational success would demonstrate resource/implementation feasibility, not validate the quality of unscaled RoPE extrapolation.\n\nOLMo at 2048 would respect its original context but would change block size, effective tokens per update, update counts and retained context relative to LLaDA. RoPE scaling or replacing OLMo with a later 4096-context generation would introduce still other changes. Report these alternatives explicitly rather than silently selecting one. This document does not authorize a full OLMo run under an undisclosed context change. [OLMo native configuration](https://huggingface.co/allenai/OLMo-7B-hf/blob/main/config.json).\n\nThe converted OLMo configuration advertises FP32 source weights. Actual shard header dtypes are audited separately. Under the common BF16-model/FP32-master procedure, native FP32 weights are rounded on BF16 model loading and FP32 masters start from that rounded model, just as the reference initializes masters from the loaded BF16 model. Preserving FP32 source masters instead would be another initialization protocol. Record this distinction if source tensors are FP32.\n\n## Controls and interpretation\n\nPreserve the same odd-index diagnostic membership as LLaDA: exclude the seven empty rows and the 1,969 exact normalized train overlaps only from diagnostic controls, yielding 3,345 clean nonempty controls. Keep all within-training duplicates. Tokenize controls with each model's native tokenizer. Near-duplicate independence is not established.\n\nThe intended production monitoring is deterministic AR CE on eight evenly spaced full training blocks and eight clean-control blocks, baseline and after each epoch, with isolated RNG state. Four repeated identical mask draws would be redundant without a diffusion corruption process, so AR diagnostics need only one deterministic pass per block. Diagnostic block identities change with tokenization; equal diagnostic article windows would require a separate documented evaluation design. Controls must not determine stopping, hyperparameters or model choice.\n\nNative-token CE and LLaDA's fixed-noise reconstruction objective are not numerically comparable across these models. This continuation comparison needs a later shared string-level memorization/extraction or membership evaluation, with the same article identities, exposure accounting, prompt/continuation definitions and duplicate controls. None of the feasibility results demonstrates memorization by itself.\n\nThe publication-date cutoff does not prove the wording was absent from any base model's pretraining data: updated/reprinted news and shared passages may predate the listed article dates. Original pretraining corpora and exposure histories differ across the models. Include an untouched native Base baseline for each model in any memorization evaluation; no objective-only causal claim about whole model families is justified by this design.\n\n## Feasibility probe and evidence boundary\n\nEach available model is loaded from its pinned original weights and independently tested at 4096 with every parameter trainable, BF16 GPU weights, non-reentrant decoder checkpointing and the exact FP32 CPU-master optimizer. The probe performs two full 16-example updates and one actual-size final-window update including the real unpadded tail. Every step requires finite loss and gradient norm, gradients for every parameter and a changed FP32 master. FP32 optimizer-state tensors and a differentiable zero-loss one-token crop are checked. Report actual peaks, not an inference-only load or a LoRA proxy.\n\nProbe-only deviations: constant learning rate `1e-5` instead of a complete production schedule; no random cropping for the full-length worst-case memory/timing measurement; only a few optimizer updates; no production epoch, diagnosis sweep or trained model checkpoint upload. A separate single-token path tests the crop edge case. These probes do not demonstrate full-run stability, inference quality or durability of a newly saved AR checkpoint. LLaDA's full-size Drive API transport tests provide prior evidence for that uploader, but new model serialization must still pass production validation.\n\nAll model downloads and token arrays are stored on the Colab local filesystem. Only small code, reports and metadata may be saved to Drive during feasibility work; no large file is copied to the Windows workspace and no probe weights are saved to Drive. A production storage plan must be based on current Drive quota and actual AR parameter counts. Retaining three snapshots for four additional models is a separate, substantially larger budget than one LLaDA run. Existing checkpoints must not be deleted to make room without the user's instruction.\n\nMeasured results and remaining questions are appended below. Machine-readable reports live in `ar_probe_results/` and are copied to Drive with the method document. When gated model access is unavailable, report the access failure rather than presenting a hardware estimate as a passed training probe.\n\n## Measured feasibility results\n\nAll sizes below are actual measurements in the existing Colab A100-SXM4-80GB runtime, which exposes 79.25 GiB VRAM and approximately 167 GiB host RAM. “Passed” means the complete three-update probe and the one-token test passed, not merely that a model loaded.\n\n| Model | 4096 full-parameter probe | Native train tokens | Updates in 3 epochs | Peak GPU allocated / reserved (GiB) | Peak process host RAM (GiB) |\n|---|---|---:|---:|---:|---:|\n| llama2_7b | blocked | — | — | — | — |\n| llama3_8b | blocked | — | — | — | — |\n| mistral7b_v01 | Passed | 4,302,687 | 198 | 43.35 / 46.75 | 111.34 |\n| olmo7b | Passed | 3,930,266 | 180 | 42.29 / 46.90 | 105.96 |\n\nLatest authenticated access check for `llama2_7b` at `2026-10-26T20:45:42.124082+00:00` returned HTTP 403: request is awaiting author review. No full-model forward/backward or optimizer test has run for this model.\n\nLatest authenticated access check for `llama3_8b` at `2026-10-26T20:45:45.161537+00:00` returned HTTP 403: request is awaiting author review. No full-model forward/backward or optimizer test has run for this model.\n\n### Pinned revisions\n\n- `meta-llama/Llama-2-7b-hf`: `01c7f73d771dfac7d292323805ebc428287df4f9`.\n- `meta-llama/Meta-Llama-3-8B`: `8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920`.\n- `mistralai/Mistral-7B-v0.1`: `27d67f1b5f57dc0953326b2601d68371d40ea8da`.\n- `allenai/OLMo-7B-hf`: `11fb3186a2e4f681edea621fa8b4345147a9db6a`.\n\n### Timing and output size\n\n| Model | Warm full update (seconds) | Approx. 3-epoch compute (hours) | One BF16 model (GB) |\n|---|---:|---:|---:|\n| mistral7b_v01 | 69.1 | 3.80 | 14.48 |\n| olmo7b | 58.3 | 2.91 | 13.78 |\n\nCompute projections use one warmed full-window measurement per model, scaled to the actual model-specific update count. They are rough estimates, not confidence intervals or total wall times. Initialization, native-token data preparation, diagnostics, local serialization, hashing, network upload and API retry time are extra. Final short accumulation windows can be faster. No trained AR weights were retained from these probes.\n\nThe probed models together require about 28.26 GB for one final BF16 snapshot each, or 84.78 GB for three epoch snapshots each, before rolling snapshots and headroom. Existing LLaDA checkpoints are additional.\n\nAt the auxiliary audit, Drive API reported 1750.55 GB (1630.33 GiB) free. This is a current account quota observation, not the mounted filesystem's reported capacity.\n\n### Independent correctness/corpus audit\n\n- `objective_matches_native_hf`: `True`.\n- `causal_future_invariance`: `True`.\n- `one_token_zero_loss`: `True`.\n- `full_and_remainder_optimizer_equivalence`: `True`.\n- `frozen_articles_reproduce_canonical_llada_token_sha256`: `f5e29db9e7b1459bf030cac1567f77f6595da59acdfaf010ab977b21fa0254df`.\n- `frozen_controls_reproduce_canonical_llada_token_sha256`: `3a6968777b4ae68e4ed1f89b3309ab4e1133a43a2a659bf7fe561dc70416e27d`.\n\nSource shard tensor dtypes, inspected without loading weight data:\n\n- `mistral7b_v01`: `{'BF16': 7241732096}`.\n- `olmo7b`: `{'F32': 6888095744}`.\n\nNative-token control stream sizes:\n\n- `mistral7b_v01`: 2,856,246 tokens across the same 3,345 clean odd-control articles.\n- `olmo7b`: 2,609,034 tokens across the same 3,345 clean odd-control articles.\n\nThe independent objective test matches the native Hugging Face shifted CE on a tiny Llama architecture; changing future inputs leaves earlier logits unchanged. The optimizer test compares full-batch CE against accumulated single-example CE across 16- and 3-example windows. These CPU checks complement actual full-model GPU probes; they do not replace those probes.\n\n### Remaining decisions / evidence limits\n\n- Whether to accept original OLMo at unscaled 4096 context rather than introduce a 2048 sequence length or a later model generation. This is an explicit scientific choice, not a capacity problem if the 4096 probe passes.\n- Durable retention for four new models given the actual Drive quota. No existing checkpoints have been removed.\n- A full production AR trainer must preserve the specified data/control/RNG/optimizer rules and validate native model serialization before long training; the probes are not a completed multi-model training pipeline.\n- Full-run stability, checkpoint loading/inference, and shared memorization evaluation remain untested by these few-update probes.\n"
Path('/content/AR_NOTEBOOK_METHOD.md').write_text(METHODS,encoding='utf-8')
print('METHODS_READY')

## 6. Small Drive write/read test and final-model quota check
Only a tiny text probe is uploaded. No base-model export or large upload test.

In [ ]:
preflight = r"""
import json
from pathlib import Path
from drive_checkpoint_upload import DriveCheckpointStore
c=json.loads(Path('/content/ar_notebook_config.json').read_text())
s=DriveCheckpointStore()
quota=s.service.about().get(fields='storageQuota').execute(num_retries=5)['storageQuota']
need=c['expected_parameters']*2+2*2**30 if c['save_final_model'] else 2**20
if quota.get('limit'):
    free=int(quota['limit'])-int(quota.get('usage',0))
    print('FREE_DRIVE_GB',round(free/1e9,2),'REQUIRED_GB',round(need/1e9,2))
    assert free>=need,'Insufficient Drive quota for the requested final output'
folder=s.ensure_path(['LLaDA-ft','ar-notebook-access-tests',c['run_name']])
p=Path('/content/ar_drive_text_probe.json')
p.write_text(json.dumps(dict(run=c['run_name'],small_write_test=True),sort_keys=True))
r=s.upload(p,folder)
assert s.service.files().get_media(fileId=r['id']).execute(num_retries=5)==p.read_bytes()
print('SMALL_DRIVE_WRITE_AND_READ_VERIFIED')
"""
subprocess.run([PYTHON,'-u','-c',preflight],check=True)

### Focused CPU checks
Checks loss alignment, one-token zero loss, schedule endpoints, final-model integrity and interrupted-upload behavior using a tiny temporary model. No GPU training or Drive model upload occurs.

In [ ]:
subprocess.run([PYTHON,'-u','/content/validate_ar_final_only.py'],check=True)

## 7. Reproduce the exact data and download the pinned base locally
Reuses the saved frozen article JSONL files if present. Otherwise rebuilds from the pinned dataset. Verifies canonical LLaDA training/control hashes, then native token counts/hashes. No token arrays or base weights are written to Drive.

In [ ]:
subprocess.run([PYTHON,'-u','/content/prepare_ar_notebook_data.py',str(CONFIG_PATH)],check=True)

## 8. Train once
This blocking cell prints live updates and writes small text logs to Drive. There are no intermediate model saves. With final saving enabled, it releases optimizer memory, serializes locally and uploads one final model after epoch 3. A final upload failure leaves local staging available while this VM survives. Do not restart training to retry an upload.

In [ ]:
assert json.loads(CONFIG_PATH.read_text())['run_name']==RUN_NAME
assert not (ROOT/'runs'/RUN_NAME).exists(),'Run already exists; inspect logs instead of restarting.'
LOG_PATH=ROOT/'logs'/(RUN_NAME+'.log')
print('RUN',RUN_NAME,'LOG',LOG_PATH,flush=True)
with LOG_PATH.open('a',encoding='utf-8',buffering=1) as log:
    process=subprocess.Popen([PYTHON,'-u','/content/train_ar_bbc.py','--config',str(CONFIG_PATH)],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            log.write(line)
            print(line,end='',flush=True)
        returncode=process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill();process.wait()
        raise
assert returncode==0,f'Training failed with exit code {returncode}; inspect the text log.'
print('Training process finished. Verify the result below.')

## 9. Verify the cloud result
Checks completion through the Drive API. With final saving enabled it verifies every saved file against server checksums and reads the completion marker. In text-only mode it explicitly reports that no model was saved.

In [ ]:
verification = r"""
import json
from pathlib import Path
from drive_checkpoint_upload import DriveCheckpointStore
c=json.loads(Path('/content/ar_notebook_config.json').read_text())
s=DriveCheckpointStore()
parent='root'
for name in ['LLaDA-ft','run-api-records',c['run_name']]:
    parent=s.folder(parent,name,create=False)
    assert parent,'Run API journal missing'
def read(name):
    items=s.children(parent,name)
    assert len(items)==1,'Missing/ambiguous journal file: '+name
    return json.loads(s.service.files().get_media(fileId=items[0]['id']).execute(num_retries=5))
status=read('status_api.json')
assert status['status']=='complete' and status['update']==c['expected_total_updates'],'Training did not complete'
if not c['save_final_model']:
    assert not status.get('final_model_verified')
    print('TEXT_ONLY_RUN_COMPLETE: NO TRAINED MODEL WAS SAVED')
else:
    receipt=read('final_upload_receipt.json')
    assert receipt['verified'] and receipt['run']==c['run_name']
    assert receipt['epoch']==3 and receipt['update']==c['expected_total_updates']
    actual={item['id']:item for item in s.children(receipt['folder_id'])}
    assert set(actual)=={item['id'] for item in receipt['files'].values()}
    for name,saved in receipt['files'].items():
        cloud=actual[saved['id']]
        assert cloud['name']==name and int(cloud['size'])==int(saved['size'])
        assert cloud['md5Checksum']==saved['verified_md5']
        if cloud.get('sha256Checksum'):
            assert cloud['sha256Checksum']==saved['local_sha256']
    marker=receipt['files']['COMPLETE.json']
    complete=json.loads(s.service.files().get_media(fileId=marker['id']).execute(num_retries=5))
    assert complete['run']==c['run_name'] and complete['epoch']==3
    assert complete['update']==c['expected_total_updates']
    assert complete['model_revision']==c['model_revision']
    print('FINAL_MODEL_CLOUD_VERIFIED',receipt['folder_url'])
print('TEXT_LOGS: My Drive/LLaDA-ft/runs/'+c['run_name'])
"""
subprocess.run([PYTHON,'-u','-c',verification],check=True)